# VectrixSync - Delta Lake to Lakebase Synchronization

VectrixSync keeps your governed Delta Lake data in sync with Lakebase for fast queries.

**Architecture:**
- **Source**: Delta Lake (Unity Catalog governed, ACID-compliant)
- **Target**: Lakebase (pgvector for fast similarity search)
- **Sync**: Automatic or manual synchronization

## Basic Setup

In [ ]:
from vectrixdb import VectrixDB, VectrixSync, StorageConfig, StorageBackend

# Configure Delta Lake (source of truth)
delta_config = StorageConfig(
    backend=StorageBackend.DELTA_LAKE,
    delta_workspace_url="https://your-workspace.cloud.databricks.com",
    delta_token="your-token",
    delta_catalog="main",
    delta_schema="vectrixdb"
)

# Configure Lakebase (fast query layer)
lakebase_config = StorageConfig(
    backend=StorageBackend.LAKEBASE,
    lakebase_host="your-lakebase-host",
    lakebase_port=5432,
    lakebase_database="vectrixdb",
    lakebase_user="user",
    lakebase_password="password"
)

# Create databases
# db_delta = VectrixDB(storage_config=delta_config)
# db_lakebase = VectrixDB(storage_config=lakebase_config)

print("Storage configs ready")

## Initialize VectrixSync

In [ ]:
# Create sync instance
# sync = VectrixSync(
#     source=db_delta,    # Delta Lake as source
#     target=db_lakebase  # Lakebase as target
# )

print("VectrixSync initialized")

## Full Sync (Initial Setup)

Run a full sync to copy all data from Delta Lake to Lakebase.

In [ ]:
# Full sync - copies everything
# result = sync.full()

# print(f"Full sync completed:")
# print(f"  Success: {result.success}")
# print(f"  Rows synced: {result.rows_synced}")
# print(f"  Collections synced: {result.collections_synced}")
# print(f"  Duration: {result.duration_seconds:.2f}s")

print("Full sync example (uncomment with real connections)")

## Incremental Sync

Sync only changes since the last sync (faster for regular updates).

In [ ]:
# Incremental sync - only new/changed data
# result = sync.incremental()

# print(f"Incremental sync:")
# print(f"  Rows synced: {result.rows_synced}")

print("Incremental sync example")

## Auto Sync (Recommended)

The easiest way: full sync if needed, then start automatic scheduler.

In [ ]:
# One-liner for automatic sync
# result = sync.auto(interval_minutes=5)

# This does:
# 1. Checks if full sync is needed
# 2. Runs full sync if never synced or data pending
# 3. Starts background scheduler for incremental syncs

# print(f"Auto sync started: {result.success}")

print("Auto sync is the recommended approach!")

## Check Sync Status

In [ ]:
# Get sync status
# status = sync.status()

# print(f"Sync Status:")
# print(f"  Last sync: {status.last_sync}")
# print(f"  Is running: {status.is_running}")
# print(f"  Collections: {status.collections}")

print("Status check example")

## Scheduled Sync

In [ ]:
# Start scheduled sync (runs in background)
# sync.start_scheduler(interval_minutes=5)

# Check if scheduler is running
# print(f"Scheduler running: {sync.is_scheduler_running()}")

# Stop scheduler when done
# sync.stop_scheduler()

print("Scheduler runs incremental syncs automatically")

## Workflow: First Time Setup

In [ ]:
# First time setup workflow:

# 1. Add data to Delta Lake
# coll = db_delta.create_collection("products", dimension=384)
# coll.add(ids=[...], vectors=[...], metadata=[...])

# 2. Create sync and run auto
# sync = VectrixSync(source=db_delta, target=db_lakebase)
# sync.auto(interval_minutes=5)  # Full sync + start scheduler

# 3. Query from Lakebase (fast!)
# results = db_lakebase.get_collection("products").search(query=[...])

print("First time: add to Delta Lake -> sync.auto() -> query Lakebase")

## Workflow: Adding New Data

In [ ]:
# After initial setup, adding new data:

# 1. Add new data to Delta Lake (source of truth)
# coll = db_delta.get_collection("products")
# coll.add(ids=["new_1"], vectors=[[...]], metadata=[{...}])

# 2. Sync happens automatically (scheduler running)
#    OR manually trigger:
# sync.incremental()

# 3. New data available in Lakebase
# results = db_lakebase.get_collection("products").search(query=[...])

print("New data: add to Delta Lake -> auto-syncs -> query Lakebase")

## Document Index Sync

VectrixSync also works with Document Index collections!

In [ ]:
from vectrixdb import DocumentIndex

# Create Document Index on Delta Lake
# doc_index = DocumentIndex(db_delta, dimension=384)
# doc_index.index_text("doc_1", content="...", metadata={...})

# Sync syncs Document Index collections too!
# sync.auto()  # Syncs both vectors and document chunks

# Query from Lakebase Document Index
# doc_index_lb = DocumentIndex(db_lakebase, dimension=384)
# results = doc_index_lb.search("query text")

print("Document Index works with sync too!")

## Why This Architecture?

**Delta Lake (Source)**
- ACID transactions
- Unity Catalog governance
- Audit trail and lineage
- Data sharing across workspaces

**Lakebase (Query Layer)**
- pgvector for fast similarity search
- Low-latency queries
- Optimized for read-heavy workloads

**VectrixSync**
- Keeps them in sync automatically
- Change data capture for efficiency
- No manual data movement

In [ ]:
architecture = """
    +-------------------+
    |   Your App        |
    +--------+----------+
             |
    +--------v----------+
    |    VectrixDB      |
    +--------+----------+
             |
    +--------+----------+--------+
    |                            |
+---v----+               +-------v-------+
|  Write |               |     Read      |
+---+----+               +-------+-------+
    |                            |
+---v---------+          +-------v-------+
| Delta Lake  |  Sync    |   Lakebase    |
| (Governed)  |--------->| (Fast Query)  |
+-------------+          +---------------+
"""
print(architecture)

## Best Practices

In [ ]:
best_practices = """
1. Always write to Delta Lake (source of truth)
2. Use sync.auto() for hands-off synchronization
3. Query from Lakebase for fast responses
4. Use incremental sync for regular updates
5. Run full sync only when recovering or initializing
6. Monitor sync status in dashboard
"""
print(best_practices)

## Cleanup

In [ ]:
# Stop scheduler before closing
# if sync.is_scheduler_running():
#     sync.stop_scheduler()

# Close connections
# db_delta.close()
# db_lakebase.close()

print("Demo complete!")